In [ ]:
import pandas as pd
import numpy as np 
import geopandas as gpd 
import matplotlib.pyplot as plt 
from matplotlib.colors import ListedColormap
from sklearn.decomposition import PCA
import seaborn as sns 
from scipy.stats import pearsonr
import json
from shapely.geometry import shape 
# from shapely.geometry import Polygon 
import json 
from shapely import wkt 
from shapely.geometry import Point
import plotly.express as px
from pandas.tseries.offsets import Week
from statsmodels.tsa.stattools import kpss 
import statsmodels.api as sm
import warnings
import scipy.stats as stats

### Read weather station data 

In [ ]:
tahmo_16 = pd.read_csv('/Users/kwamedonkor/Downloads/UW/Research/TAHMO/With_Lightning_Distance/TAHMO_export_673b722af8aada2ee55cef6b/TA00016.csv')
tahmo_98 = pd.read_csv('/Users/kwamedonkor/Downloads/UW/Research/TAHMO/With_Lightning_Distance/TAHMO_export_673b722af8aada2ee55cef6b/TA00098.csv')
tahmo_118 = pd.read_csv('/Users/kwamedonkor/Downloads/UW/Research/TAHMO/With_Lightning_Distance/TAHMO_export_673b722af8aada2ee55cef6b/TA00118.csv')

tahmo_126 = pd.read_csv('/Users/kwamedonkor/Downloads/UW/Research/TAHMO/With_Lightning_Distance/TAHMO_export_673b722af8aada2ee55cef6b/TA00126.csv')
tahmo_127 = pd.read_csv('/Users/kwamedonkor/Downloads/UW/Research/TAHMO/With_Lightning_Distance/TAHMO_export_673b722af8aada2ee55cef6b/TA00127.csv')
tahmo_313 = pd.read_csv('/Users/kwamedonkor/Downloads/UW/Research/TAHMO/With_Lightning_Distance/TAHMO_export_673b722af8aada2ee55cef6b/TA00313.csv')
tahmo_319 = pd.read_csv('/Users/kwamedonkor/Downloads/UW/Research/TAHMO/With_Lightning_Distance/TAHMO_export_673b722af8aada2ee55cef6b/TA00319.csv')
tahmo_391 = pd.read_csv('/Users/kwamedonkor/Downloads/UW/Research/TAHMO/With_Lightning_Distance/TAHMO_export_673b722af8aada2ee55cef6b/TA00391.csv')
tahmo_567 = pd.read_csv('/Users/kwamedonkor/Downloads/UW/Research/TAHMO/With_Lightning_Distance/TAHMO_export_673b722af8aada2ee55cef6b/TA00567.csv')
tahmo_647 = pd.read_csv('/Users/kwamedonkor/Downloads/UW/Research/TAHMO/With_Lightning_Distance/TAHMO_export_673b722af8aada2ee55cef6b/TA00647.csv')
tahmo_651 = pd.read_csv('/Users/kwamedonkor/Downloads/UW/Research/TAHMO/With_Lightning_Distance/TAHMO_export_673b722af8aada2ee55cef6b/TA00651.csv')

### Filter Dataframes (select relevant columns)

In [ ]:
def filter_columns(df):

    columns_to_keep = ['timestamp', 'lightningdistance (km)', 'lightningevents (-)', 'precipitation (mm)', 'temperature (degrees Celsius)', 'windgusts (m/s)', 'windspeed (m/s)']
    
    # Filter the DataFrame to keep only the specified columns
    filtered_df = df[columns_to_keep].copy()

    filtered_df['timestamp'] = pd.to_datetime(filtered_df['timestamp'])
        
    # Define a dictionary for renaming columns
    rename_mapping = {
        'timestamp': 'Timestamp',
        'lightningdistance (km)': 'Lightning Distance', 
        'lightningevents (-)': 'Lightning Events',
        'precipitation (mm)': 'Precipitation (mm)',
        'temperature (degrees Celsius)': 'Temperature (°C)',
        'windgusts (m/s)': 'Wind Gusts (m/s)',
        'windspeed (m/s)': 'Wind Speed (m/s)'
    }

    # Rename the columns using the mapping
    filtered_df = filtered_df.rename(columns=rename_mapping)

    return filtered_df

In [ ]:
def filter_columns_rev3(df):

    columns_to_keep = ['timestamp', 'lightningdistance (km)', 'lightningevents (-)', 'precipitation S001265 (mm)', 'temperature (degrees Celsius)', 'windgusts (m/s)', 'windspeed (m/s)']
    
    # Filter the DataFrame to keep only the specified columns
    filtered_df = df[columns_to_keep].copy()

    filtered_df['timestamp'] = pd.to_datetime(filtered_df['timestamp'])

    # Define a dictionary for renaming columns
    rename_mapping = {
        'timestamp': 'Timestamp',
        'lightningdistance (km)': 'Lightning Distance', 
        'lightningevents (-)': 'Lightning Events',
        'precipitation S001265 (mm)': 'Precipitation (mm)',
        'temperature (degrees Celsius)': 'Temperature (°C)',
        'windgusts (m/s)': 'Wind Gusts (m/s)',
        'windspeed (m/s)': 'Wind Speed (m/s)'
    }
    
    # Rename the columns using the mapping
    filtered_df = filtered_df.rename(columns=rename_mapping)

    return filtered_df

### 5 min re-index function 

In [ ]:
def fill_missing_timestamps(df, timestamp_col='Timestamp', freq='5min'):
    if timestamp_col in df.columns:
        df = df.copy()  # Avoid SettingWithCopyWarning

        # Convert to datetime and set index
        df[timestamp_col] = pd.to_datetime(df[timestamp_col])
        df.set_index(timestamp_col, inplace=True)

        # Generate complete timestamp range
        full_range = pd.date_range(start=df.index.min(), end=df.index.max(), freq=freq)
        original_len = len(df)
        df_reindexed = df.reindex(full_range)
        df_reindexed.index.name = timestamp_col

        # Print NaN summary
        total_entries = df_reindexed.shape[0] * df_reindexed.shape[1]
        total_nans = df_reindexed.isna().sum().sum()
        nan_percent = (total_nans / total_entries) * 100 if total_entries > 0 else 0

        print(f"\nDataFrame:")
        print(f"- Original length: {original_len}")
        print(f"- After reindexing: {len(df_reindexed)} rows")
        print(f"- Total NaNs: {total_nans:,} ({nan_percent:.2f}%)")

        return df_reindexed.reset_index()
    
    else:
        print(f"Timestamp column '{timestamp_col}' not found in DataFrame.")
        return df

### Linear Interpolation Function 

In [ ]:
def interpolate_cleaned_df_linear_multi(df, cols=['Wind Gusts (m/s)', 'Wind Speed (m/s)']):
    df_interp = df.copy()

    # Set datetime index if needed
    if 'Timestamp' in df_interp.columns:
        df_interp = df_interp.set_index('Timestamp')

    if not isinstance(df_interp.index, pd.DatetimeIndex):
        raise ValueError("DataFrame index must be a DatetimeIndex")

    # Normalize date once
    normalized_index = df_interp.index.normalize()

    for col in cols:
        # Create new interpolated column
        new_col = f'Interpolated {col}'
        df_interp[new_col] = df_interp[col]

        # Compute % of NaNs per day for this column
        nan_percent = (
            df_interp[col]
            .isna()
            .groupby(normalized_index)
            .mean() * 100
        )

        # Add NaN_Percent_Per_Day column for this variable
        df_interp[f'NaN_Percent_Per_Day_{col}'] = normalized_index.map(nan_percent)

    # Group by day
    grouped = df_interp.groupby(normalized_index)

    interpolated_groups = []
    for _, group in grouped:
        for col in cols:
            new_col = f'Interpolated {col}'
            group[new_col] = group[new_col].interpolate(method='linear', limit_direction='both')
        interpolated_groups.append(group)

    return pd.concat(interpolated_groups)


In [ ]:
### Filter Weather Station data to select relevant columns 


# rev1 
accra_aca_df = filter_columns(tahmo_16)
g_met_hq_df = filter_columns(tahmo_98)
temasco_df = filter_columns(tahmo_118)
st_johns_df = filter_columns(tahmo_127)
safisana_df = filter_columns(tahmo_313)

nsawam_df = filter_columns(tahmo_319)
agri_impact_df = filter_columns(tahmo_391)
accra_girls_df = filter_columns(tahmo_567)
legon_df = filter_columns(tahmo_647)
madina_df = filter_columns(tahmo_651)


##rev3 
berekuso_df = filter_columns_rev3(tahmo_126)

### Legon - Wind 

In [ ]:
legon_df_wind = legon_df[['Timestamp', 'Wind Gusts (m/s)', 'Wind Speed (m/s)']]

# re-index data 
legon_df_wind = fill_missing_timestamps(legon_df_wind, timestamp_col='Timestamp', freq='5min')

# run linear interpolation 
legon_df_wind = interpolate_cleaned_df_linear_multi(legon_df_wind)

# Resample by hour & select avg for wind speed & max for wind gusts 
hourly_wind_legon = legon_df_wind.resample('1H').agg({
    'Interpolated Wind Speed (m/s)': 'mean',
    'Interpolated Wind Gusts (m/s)': 'max'
}).reset_index()

# select location & study period 
hourly_wind_legon['Location'] = 'Legon'
hourly_wind_legon = hourly_wind_legon.set_index('Timestamp').loc['2022':'2023'].reset_index()

### Madina - Wind 

In [ ]:
madina_df_wind = madina_df[['Timestamp', 'Wind Gusts (m/s)', 'Wind Speed (m/s)']]

# re-index data 
madina_df_wind = fill_missing_timestamps(madina_df_wind, timestamp_col='Timestamp', freq='5min')

# run linear interpolation 
madina_df_wind = interpolate_cleaned_df_linear_multi(madina_df_wind)

# Resample by hour & select avg for wind speed & max for wind gusts 
hourly_wind_madina = madina_df_wind.resample('1H').agg({
    'Interpolated Wind Speed (m/s)': 'mean',
    'Interpolated Wind Gusts (m/s)': 'max'
}).reset_index()

# select location & study period 
hourly_wind_madina['Location'] = 'Madina'
hourly_wind_madina = hourly_wind_madina.set_index('Timestamp').loc['2022':'2023'].reset_index()


### G_Met - Wind 

In [ ]:
g_met_df_wind = g_met_hq_df[['Timestamp', 'Wind Gusts (m/s)', 'Wind Speed (m/s)']]

g_met_df_wind = fill_missing_timestamps(g_met_df_wind, timestamp_col='Timestamp', freq='5min')

g_met_df_wind = interpolate_cleaned_df_linear_multi(g_met_df_wind)

# Resample by hour 
hourly_wind_g_met = g_met_df_wind.resample('1H').agg({
    'Interpolated Wind Speed (m/s)': 'mean',
    'Interpolated Wind Gusts (m/s)': 'max'
}).reset_index()

hourly_wind_g_met['Location'] = 'G_Met'
hourly_wind_g_met = hourly_wind_g_met.set_index('Timestamp').loc['2022':'2023'].reset_index()

### Agri_Impact - Wind 

In [ ]:
agri_impact_wind = agri_impact_df[['Timestamp', 'Wind Gusts (m/s)', 'Wind Speed (m/s)']]

agri_impact_wind = fill_missing_timestamps(agri_impact_wind, timestamp_col='Timestamp', freq='5min')

agri_impact_wind = interpolate_cleaned_df_linear_multi(agri_impact_wind)

# Resample by hour 
hourly_wind_agri_impact = agri_impact_wind.resample('1H').agg({
    'Interpolated Wind Speed (m/s)': 'mean',
    'Interpolated Wind Gusts (m/s)': 'max'
}).reset_index()

hourly_wind_agri_impact['Location'] = 'Agri_Impact'
hourly_wind_agri_impact = hourly_wind_agri_impact.set_index('Timestamp').loc['2022':'2023'].reset_index()

### Berekuso - Wind 

In [ ]:
berekuso_wind = berekuso_df[['Timestamp', 'Wind Gusts (m/s)', 'Wind Speed (m/s)']]

berekuso_wind = fill_missing_timestamps(berekuso_wind, timestamp_col='Timestamp', freq='5min')

berekuso_wind = interpolate_cleaned_df_linear_multi(berekuso_wind)

# berekuso_wind = berekuso_wind.set_index('Timestamp')

# Resample by hour 
hourly_wind_berekuso = berekuso_wind.resample('1H').agg({
    'Interpolated Wind Speed (m/s)': 'mean',
    'Interpolated Wind Gusts (m/s)': 'max'
}).reset_index()

hourly_wind_berekuso['Location'] = 'Berekuso'
hourly_wind_berekuso = hourly_wind_berekuso.set_index('Timestamp').loc['2022':'2023'].reset_index()

### Temasco - Wind 

In [ ]:
temasco_wind = temasco_df[['Timestamp', 'Wind Gusts (m/s)', 'Wind Speed (m/s)']]

temasco_wind = fill_missing_timestamps(temasco_wind, timestamp_col='Timestamp', freq='5min')

temasco_wind = interpolate_cleaned_df_linear_multi(temasco_wind)

# Resample by hour 
hourly_wind_temasco = temasco_wind.resample('1H').agg({
    'Interpolated Wind Speed (m/s)': 'mean',
    'Interpolated Wind Gusts (m/s)': 'max'
}).reset_index()

hourly_wind_temasco['Location'] = 'Temasco'
hourly_wind_temasco = hourly_wind_temasco.set_index('Timestamp').loc['2022':'2023'].reset_index()

### Safisana - Wind 

In [ ]:
safisana_wind = safisana_df[['Timestamp', 'Wind Gusts (m/s)', 'Wind Speed (m/s)']]

safisana_wind = fill_missing_timestamps(safisana_wind, timestamp_col='Timestamp', freq='5min')

safisana_wind = interpolate_cleaned_df_linear_multi(safisana_wind)


# Resample by hour 
hourly_wind_safisana = safisana_wind.resample('1H').agg({
    'Interpolated Wind Speed (m/s)': 'mean',
    'Interpolated Wind Gusts (m/s)': 'max'
}).reset_index()

hourly_wind_safisana['Location'] = 'Safisana'
hourly_wind_safisana = hourly_wind_safisana.set_index('Timestamp').loc['2022':'2023'].reset_index()


### St_Johns - Wind 

In [ ]:
st_johns_wind = st_johns_df[['Timestamp', 'Wind Gusts (m/s)', 'Wind Speed (m/s)']]

st_johns_wind = fill_missing_timestamps(st_johns_wind, timestamp_col='Timestamp', freq='5min')

st_johns_wind = interpolate_cleaned_df_linear_multi(st_johns_wind)

# Resample by hour 
hourly_wind_st_johns = st_johns_wind.resample('1H').agg({
    'Interpolated Wind Speed (m/s)': 'mean',
    'Interpolated Wind Gusts (m/s)': 'max'
}).reset_index()

hourly_wind_st_johns['Location'] = 'St_Johns'
hourly_wind_st_johns = hourly_wind_st_johns.set_index('Timestamp').loc['2022':'2023'].reset_index()

### Accra_Aca - Wind 

In [ ]:
accra_aca_wind = accra_aca_df[['Timestamp', 'Wind Gusts (m/s)', 'Wind Speed (m/s)']]

accra_aca_wind = fill_missing_timestamps(accra_aca_wind, timestamp_col='Timestamp', freq='5min')

accra_aca_wind = interpolate_cleaned_df_linear_multi(accra_aca_wind)

# Resample by hour 
hourly_wind_accra_aca = accra_aca_wind.resample('1H').agg({
    'Interpolated Wind Speed (m/s)': 'mean',
    'Interpolated Wind Gusts (m/s)': 'max'
}).reset_index()

hourly_wind_accra_aca['Location'] = 'Accra_Aca'
hourly_wind_accra_aca = hourly_wind_accra_aca.set_index('Timestamp').loc['2022':'2023'].reset_index()

### Accra_Girls - Wind 

In [ ]:
accra_girls_wind = accra_girls_df[['Timestamp', 'Wind Gusts (m/s)', 'Wind Speed (m/s)']]

accra_girls_wind = fill_missing_timestamps(accra_girls_wind, timestamp_col='Timestamp', freq='5min')

accra_girls_wind = interpolate_cleaned_df_linear_multi(accra_girls_wind)

# Resample by hour 
hourly_wind_accra_girls = accra_girls_wind.resample('1H').agg({
    'Interpolated Wind Speed (m/s)': 'mean',
    'Interpolated Wind Gusts (m/s)': 'max'
}).reset_index()

hourly_wind_accra_girls['Location'] = 'Accra_Girls'
hourly_wind_accra_girls = hourly_wind_accra_girls.set_index('Timestamp').loc['2022':'2023'].reset_index()

### Nsawam - Wind 

In [ ]:
nsawam_wind = nsawam_df[['Timestamp', 'Wind Gusts (m/s)', 'Wind Speed (m/s)']]

nsawam_wind = fill_missing_timestamps(nsawam_wind, timestamp_col='Timestamp', freq='5min')

nsawam_wind = interpolate_cleaned_df_linear_multi(nsawam_wind)

# Resample by hour 
hourly_wind_nsawam = nsawam_wind.resample('1H').agg({
    'Interpolated Wind Speed (m/s)': 'mean',
    'Interpolated Wind Gusts (m/s)': 'max'
}).reset_index()

hourly_wind_nsawam['Location'] = 'Nsawam'
hourly_wind_nsawam = hourly_wind_nsawam.set_index('Timestamp').loc['2022':'2023'].reset_index()


### Concatenate - All Weather Station data 

In [ ]:
# Excluded temasco, berekuso, g_met, accra_aca 
# The rest have mostly full year data 

dfs = [hourly_wind_legon, hourly_wind_madina, hourly_wind_accra_girls, hourly_wind_st_johns, hourly_wind_safisana, hourly_wind_nsawam, hourly_wind_agri_impact]

combined_wind_df = pd.concat(dfs, ignore_index=True)

combined_wind_df = combined_wind_df.rename(columns={
    'Interpolated Wind Speed (m/s)': 'Wind Speed (m/s)',
    'Interpolated Wind Gusts (m/s)': 'Wind Gusts (m/s)'
})

# combined_wind_df.to_csv('/Users/kwamedonkor/Downloads/UW/Research/Quals/Output_Files/combined_wind_df.csv')

### Comparison Plots (Justification for use of 6km -> doesn't change that much over 6km) 

#### 

In [ ]:
# excluded the stations with lots of NaNs 

In [ ]:
def resample_and_plot_wind_multi(df1, df2=None, df3=None, freq='H', title='', labels=None):
    def prepare(df):
        df = df.copy()
        df['Timestamp'] = pd.to_datetime(df['Timestamp'])
        df.set_index('Timestamp', inplace=True)
        return df['Interpolated Wind Gusts (m/s)'].resample(freq).mean()

    # Prepare and collect all valid DataFrames
    series_list = [prepare(df1)]
    if df2 is not None:
        series_list.append(prepare(df2))
    if df3 is not None:
        series_list.append(prepare(df3))

    if labels is None:
        labels = [f"Dataset {i+1}" for i in range(len(series_list))]

    # Plot
    plt.figure(figsize=(14, 6))
    for series, label in zip(series_list, labels):
        series.plot(label=label)

    plt.title(title)
    plt.xlabel('Time')
    plt.ylabel('Average Wind Gusts (m/s)', labelpad=10)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    return

In [ ]:
def print_wind_stats(df, column='Interpolated Wind Gusts (m/s)'):
    mean_val = df[column].mean()
    std_val = df[column].std()
    print(f"Mean of {column}: {mean_val:.2f}")
    print(f"Standard Deviation of {column}: {std_val:.2f}")

In [ ]:
def calc_wind_corr(df1, df2, column='Interpolated Wind Gusts (m/s)'):
    common_index = df1.index.intersection(df2.index)
    series1 = df1.loc[common_index, column]
    series2 = df2.loc[common_index, column]
    corr = series1.corr(series2)
    print(f"Correlation between {column} in the two DataFrames: {corr:.3f}")
    return 

In [ ]:
resample_and_plot_wind_multi(
    df1=hourly_wind_madina,
    df2=hourly_wind_legon,
    title='Daily Rainfall Comparison - Madina & Legon',
    labels=['Madina', 'Legon']
)

In [ ]:
print_wind_stats(hourly_wind_madina)
print("")
print_wind_stats(hourly_wind_legon)

In [ ]:
calc_wind_corr(hourly_wind_madina, hourly_wind_legon)

In [ ]:
resample_and_plot_wind_multi(
    df1=hourly_wind_accra_girls,
    df2=hourly_wind_st_johns,
    title='Hourly Comparison - Accra_Girls & St_Johns',
    labels=['Accra_Girls', 'St_Johns']
)

In [ ]:
print_wind_stats(hourly_wind_accra_girls)
print("")
print_wind_stats(hourly_wind_st_johns)

In [ ]:
calc_wind_corr(hourly_wind_accra_girls, hourly_wind_st_johns)